# 13 — Differential Abundance Testing with Milo

**Tool:** `milopy` (Milo — Milotools)  
**Dataset:** Norman 2019 CRISPRa (`data/norman_assigned.h5ad`)  
**Paper:** [Dann et al. 2022, Nature Biotechnology](https://doi.org/10.1038/s41587-021-01033-z)

**Prerequisites:** Run notebook 07 first. Requires `../data/norman_assigned.h5ad`.

---

## What is differential abundance?

DE analysis (notebook 08) asks: *which genes change when we perturb gene X?*  
Differential abundance asks: *which cell states become more or less prevalent when we perturb gene X?*

These are complementary questions. A perturbation might:
1. Change gene expression within a cell state (detected by DE)
2. Shift cells from one state to another (detected by DA)
3. Both

## Why Milo over a simple χ² test on cluster fractions?

Committing to clusters (Leiden, Louvain) is lossy — cells on cluster boundaries are forced into one label. Milo avoids this by:
1. Defining **kNN neighborhoods** (overlapping sets of nearest neighbors per cell)
2. Counting how many cells from each condition fall in each neighborhood
3. Testing DA at the neighborhood level using a **quasi-binomial GLM** (edgeR-style)

Neighborhoods near cluster centers get high counts; boundaries get tested too. This gives continuous, boundary-agnostic DA testing.

```
  NTC cells + Perturbed cells
       ↓  build kNN graph
  Neighborhoods (N_i = set of k nearest cells to cell i)
       ↓  count: how many NTC vs. perturbed in N_i?
  GLM: count_perturbed ~ condition + log(total)
       ↓
  log-fold change and FDR per neighborhood
       ↓
  Spatial FDR correction (accounting for graph structure)
```

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import milopy
import milopy.core as milo
import milopy.utils as milo_utils
import matplotlib.pyplot as plt

sc.settings.set_figure_params(dpi=80, facecolor='white')
print(f'milopy {milopy.__version__}')

## 1. Load Data

Milo needs:
- A **preprocessed, embedded** AnnData (PCA + neighbors graph already built)
- A **sample/replicate** column in `.obs` — Milo tests across samples, not cells
- A **condition** column to compare

In [ ]:
adata = sc.read_h5ad('../data/norman_assigned.h5ad')
print(adata)
print('\nobs columns:', adata.obs.columns.tolist())

In [ ]:
# Select a subset of perturbations for a focused DA analysis
# Compare: NTC cells vs. cells perturbed with top effectors from notebook 08
top_perts = ['CBX2', 'SET', 'RUNX1', 'CEBPA', 'KLF1']
mask = adata.obs['gene'].isin(['NTC'] + top_perts)
adata_sub = adata[mask].copy()
print(f'Subsetting to {adata_sub.n_obs} cells')
print(adata_sub.obs['gene'].value_counts())

In [ ]:
# Add a 'condition' binary column: NTC vs. perturbed
adata_sub.obs['condition'] = adata_sub.obs['gene'].apply(
    lambda g: 'NTC' if g == 'NTC' else 'perturbed'
).astype('category')

# Milo requires a 'sample' column (replicate identifier)
# For demonstration, simulate 4 replicates by randomly assigning cells
np.random.seed(42)
adata_sub.obs['sample'] = np.random.choice(
    ['rep1', 'rep2', 'rep3', 'rep4'], size=adata_sub.n_obs
).astype(str)

print('Sample distribution:')
print(pd.crosstab(adata_sub.obs['condition'], adata_sub.obs['sample']))

In [ ]:
# Ensure the KNN graph is built (Milo uses it to define neighborhoods)
if 'neighbors' not in adata_sub.uns:
    sc.pp.normalize_total(adata_sub, target_sum=1e4)
    sc.pp.log1p(adata_sub)
    sc.pp.highly_variable_genes(adata_sub, n_top_genes=2000)
    sc.pp.pca(adata_sub)
    sc.pp.neighbors(adata_sub, n_neighbors=30)
    sc.tl.umap(adata_sub)

print('Neighbors graph:', adata_sub.obsp['connectivities'].shape)

## 2. Milo: Build Neighborhoods and Test DA

In [ ]:
# Step 1: Make Milo object (wraps AnnData)
milo_data = milo.Milo(adata_sub)
print(milo_data)

In [ ]:
# Step 2: Sample index cells (representative cells that define neighborhoods)
# prop: fraction of cells to use as neighborhood centers
# d: number of PCs to use for refinement
milo.make_nhoods(milo_data, prop=0.1)
print(f'Number of neighborhoods: {milo_data.nhoods.shape[1]}')

In [ ]:
# Step 3: Count cells per sample in each neighborhood
# sample_col: the replicate/sample column
milo.count_nhoods(milo_data, sample_col='sample')
print('Neighborhood count matrix shape:', milo_data.uns['nhood_counts'].shape)

In [ ]:
# Step 4: Build the experimental design matrix
# Maps sample → condition
design_df = adata_sub.obs[['sample', 'condition']].drop_duplicates().set_index('sample')
print('Design matrix:')
print(design_df)

In [ ]:
# Step 5: DA testing
# Uses quasi-binomial GLM (as in edgeR) per neighborhood
# Applies spatial FDR correction (accounts for graph-correlated tests)
milo.DA_nhoods(
    milo_data,
    design=design_df,
    model_contrasts='conditionperturbed - conditionNTC',
)

# Results: per-neighborhood logFC + SpatialFDR
results = milo_data.uns['da_results']
print('DA results columns:', results.columns.tolist())
print('\nSignificant neighborhoods (SpatialFDR < 0.1):')
print((results['SpatialFDR'] < 0.1).sum(), 'of', len(results))

## 3. Visualize DA Results

In [ ]:
# Volcano plot: logFC vs. -log10(SpatialFDR)
fig, ax = plt.subplots(figsize=(6, 4))
results['neg_log_fdr'] = -np.log10(results['SpatialFDR'].clip(1e-10))

colors = ['#d62728' if row['SpatialFDR'] < 0.1 and row['logFC'] > 0
          else '#1f77b4' if row['SpatialFDR'] < 0.1 and row['logFC'] < 0
          else 'lightgray'
          for _, row in results.iterrows()]

ax.scatter(results['logFC'], results['neg_log_fdr'], c=colors, s=10, alpha=0.7)
ax.axhline(-np.log10(0.1), color='black', linestyle='--', linewidth=0.8, label='SpatialFDR=0.1')
ax.axvline(0, color='gray', linewidth=0.5)
ax.set_xlabel('logFC (perturbed vs. NTC)')
ax.set_ylabel('-log10(SpatialFDR)')
ax.set_title('Milo: Differential Abundance')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Project DA results onto UMAP
# Color each neighborhood center by logFC; size by -log10(SpatialFDR)
milo.plot_nhood_graph(
    milo_data,
    alpha=0.1,       # SpatialFDR threshold for highlighting
    min_logFC=0.5,   # minimum absolute logFC to show
    figsize=(6, 5),
)

In [ ]:
# Annotate neighborhoods with cluster labels
# Which cell types (Leiden clusters) are DA?
milo.annotate_nhoods(milo_data, anno_col='leiden' if 'leiden' in adata_sub.obs else 'gene')

# Beeswarm: distribution of logFC per annotated cluster
milo.plot_DA_beeswarm(
    milo_data,
    alpha=0.1,
    subset_nhoods=results.index,
)

## 4. Per-Perturbation DA (Many vs. NTC)

Run Milo for each perturbation independently to get a DA profile per gene KO.

In [ ]:
da_summary = {}
for pert in top_perts:
    # Subset to this perturbation + NTC
    mask_p = adata_sub.obs['gene'].isin(['NTC', pert])
    ad_p = adata_sub[mask_p].copy()
    ad_p.obs['condition_p'] = (ad_p.obs['gene'] == pert).astype(int)
    
    # Quick Milo run
    md = milo.Milo(ad_p)
    milo.make_nhoods(md, prop=0.1)
    milo.count_nhoods(md, sample_col='sample')
    
    design_p = ad_p.obs[['sample', 'condition_p']].drop_duplicates().set_index('sample')
    try:
        milo.DA_nhoods(md, design=design_p, model_contrasts='condition_p')
        res_p = md.uns['da_results']
        n_sig = (res_p['SpatialFDR'] < 0.1).sum()
        mean_lfc = res_p.loc[res_p['SpatialFDR'] < 0.1, 'logFC'].mean()
        da_summary[pert] = {'n_DA_nhoods': n_sig, 'mean_logFC': mean_lfc}
    except Exception as e:
        da_summary[pert] = {'n_DA_nhoods': 0, 'mean_logFC': np.nan}
        print(f'{pert}: {e}')

da_df = pd.DataFrame(da_summary).T.sort_values('n_DA_nhoods', ascending=False)
print('DA summary per perturbation:')
print(da_df)

---
## Summary

```python
# Milo workflow
import milopy.core as milo

milo_data = milo.Milo(adata)                         # 1. wrap AnnData
milo.make_nhoods(milo_data, prop=0.1)                # 2. define neighborhoods
milo.count_nhoods(milo_data, sample_col='replicate') # 3. count cells per sample
milo.DA_nhoods(milo_data, design=design_df,          # 4. test
               model_contrasts='conditionKO - conditionNTC')
# results in milo_data.uns['da_results']
```

**Milo vs. scCODA (T08):**
- Milo: cell-level, kNN neighborhood resolution, best for finding spatially continuous gradients
- scCODA: sample-level, cluster-resolution, best for whole-cell-type compositional shifts

**Next:** 14 — batch correction for multi-sample Perturb-seq experiments.